# Autograd
Autograd is a core component of PyTorch that provides automatic differentiation for all operations on tensors. It allows computing gradients automatically, which is essential for training neural networks. When operations are performed on tensors, PyTorch builds a computational graph that tracks the operations and their dependencies. When `backward()` is called on a tensor, PyTorch traverses this graph to compute the gradients(derivatives) of loss function with respect to model parameters. This makes it easy to implement backpropagation and optimize model parameters using gradient descent.

## Computational Graphs
A computational graph is a directed acyclic graph(DAG) that represents the sequence of operations performed on tensors. Each node in the graph represents an operation, and the edges represent the flow of data (tensors) between operations. When an operation is performed on tensors, PyTorch automatically constructs this graph. The graph is dynamic, meaning it can change with each iteration of training, which allows for flexibility in model design. DAG or Directed Acyclic Graph is a graph that is directed and contains no cycles. In the context of computational graphs, it means that the graph has a direction (from inputs to outputs) and does not contain any loops or cycles, which ensures that the computations can be performed in a well-defined order. This way PyTorch can track the operations and their dependencies, allowing for efficient computation of gradients during backpropagation.

Let a function $f(x,y,z) = 5 \times (xy + z)$. For easy understanding, let $xy = u$, $xy + z = u + z = v$, and $5 \times (xy + z) = 5v$.
The computational graph for this function would look like this:

```graph
    x     y     z
     \   /     /
      \ /     /
       u     /
        \   /
         \ /
          v 
           \
           *5
             \
             f(x,y,z)       (output)
```

In [1]:
import torch

# Let we have a functions y=x^2 and z=sin(y), we want to compute dz/dx. To do so we have to backpropagate by chain rule: dz/dx = dz/dy * dy/dx. In PyTorch, we can compute this using autograd as follows:

# Create a tensor with requires_grad=True to track computations
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
z = torch.sin(y)

# Compute the gradients
z.backward()        # This will compute the gradients of z with respect to x
print(f"dz/dx: {x.grad}")  # This will give us the value of dz/dx at x=2.0

dz/dx: -2.614574432373047


### Autograd Core
The steps for a neural network training loop using autograd typically involve:
1. Forward Pass
    - Linear Transformation: $Z = W\cdot X + b$
    - Activation Function: $\hat{y} = \sigma(Z) = \frac{1}{1 + e^{-Z}}$
    - Loss Calculation(Binary Cross Entropy-BCE): $L = \text{loss}(\hat{y}, y) = -\frac{1}{n}\sum_{i=1}^{n} [y_i \log(\hat{y}_i) + (1-y_i) \log(1-\hat{y}_i)]$
2. Backward Pass
    - Compute Gradients: $\frac{\partial L}{\partial \hat{y}}$, $\frac{\partial \hat{y}}{\partial Z}$, $\frac{\partial Z}{\partial W}$, $\frac{\partial Z}{\partial b}$, $\frac{\partial Z}{\partial X}$
3. Parameter Update
    - Update Weights: $W = W - \alpha \cdot \frac{\partial L}{\partial W}$
    - Update Biases: $b = b - \alpha \cdot \frac{\partial L}{\partial b}$

Where:
- $X$ is the input data
- $W$ is the weight matrix
- $b$ is the bias vector
- $Z$ is the linear transformation output
- $\hat{y}$ is the predicted output after applying the activation function
- $L$ is the loss calculated using a loss function (e.g., binary cross-entropy)
- $\alpha$ is the learning rate

Calculate $\frac{\partial L}{\partial \hat{y}}$:
$$\frac{\partial L}{\partial \hat{y}} = -\frac{1}{n} \sum_{i=1}^{n} \left( \frac{y_i}{\hat{y}_i} - \frac{1-y_i}{1-\hat{y}_i} \right)$$
Calculate $\frac{\partial \hat{y}}{\partial Z}$: Same as the derivative of sigmoid function:
$$\frac{\partial \hat{y}}{\partial Z} = \hat{y} \cdot (1 - \hat{y})$$
Calculate $\frac{\partial Z}{\partial W}$:
$$\frac{\partial Z}{\partial W} = \frac{\partial (W \cdot X + b)}{\partial W} = X$$
Calculate $\frac{\partial Z}{\partial b}$:
$$\frac{\partial Z}{\partial b} = \frac{\partial (W \cdot X + b)}{\partial b} = 1$$
Calculate $\frac{\partial Z}{\partial X}$:
$$\frac{\partial Z}{\partial X} = \frac{\partial (W \cdot X + b)}{\partial X} = W$$
Calculate $\frac{\partial L}{\partial W}$:
$$\frac{\partial L}{\partial W} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial Z} \cdot \frac{\partial Z}{\partial W} = -\frac{1}{n} \sum_{i=1}^{n} \left( \frac{y_i}{\hat{y}_i} - \frac{1-y_i}{1-\hat{y}_i} \right) \cdot \hat{y} \cdot (1 - \hat{y}) \cdot X$$
Calculate $\frac{\partial L}{\partial b}$:
$$\frac{\partial L}{\partial b} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial Z} \cdot \frac{\partial Z}{\partial b} = -\frac{1}{n} \sum_{i=1}^{n} \left( \frac{y_i}{\hat{y}_i} - \frac{1-y_i}{1-\hat{y}_i} \right) \cdot \hat{y} \cdot (1 - \hat{y})$$
Calculate $\frac{\partial L}{\partial X}$:
$$\frac{\partial L}{\partial X} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial Z} \cdot \frac{\partial Z}{\partial X} =  -\frac{1}{n} \sum_{i=1}^{n} \left( \frac{y_i}{\hat{y}_i} - \frac{1-y_i}{1-\hat{y}_i} \right)  \cdot \hat{y} \cdot (1 - \hat{y}) \cdot W$$

Computational Graph of the above calculations:

```graph
    W     X     b                y
     \   /     /                /
      \ /     /                /
       *     /                /
        \   /                /
         \ /                /
          +                /
           \              /
            \            /
             Z          / (linear transformation)
              \        /
               \      /
                σ    /  (activation function)
                 \  / 
                  \/ 
                y_pred
                   |
                   L       (loss)
```

In [49]:
import torch

data = torch.tensor(
    [
        [2,8,75],
        [3,7,80],
        [4,6,85]
    ]
).float()

X = data[:, :2]  # Features (first two columns)
y = data[:, 2]  # Target variable (last column)

In [2]:
X

tensor([[2., 8.],
        [3., 7.],
        [4., 6.]])

In [3]:
y

tensor([75., 80., 85.])

In [50]:
# Initialize weights and bias with tracking for gradients
W = torch.tensor([[1,1],[1,1]], requires_grad=True, dtype=torch.float)
b = torch.tensor(0.0, requires_grad=True)

In [16]:
W

tensor([[1., 1.],
        [1., 1.]], requires_grad=True)

In [13]:
b

tensor(0., requires_grad=True)

In [51]:
z= torch.matmul(X, W) + b
z

tensor([[10., 10.],
        [10., 10.],
        [10., 10.]], grad_fn=<AddBackward0>)

In [52]:
y_pred = z

loss = torch.mean((y_pred - y.view(-1, 1))**2) # MSE Loss
print(f"Loss: {loss.item()}")

loss.backward()  # Compute gradients
print(f"Gradient of W: {W.grad}")
print(f"Gradient of b: {b.grad}")

# Clear gradients for next iteration
W.grad.zero_()
b.grad.zero_()

Loss: 4916.66650390625
Gradient of W: tensor([[-213.3333, -213.3333],
        [-486.6667, -486.6667]])
Gradient of b: -140.00001525878906


tensor(0.)

#### Clear Gradients
After each iteration of training, it's important to clear the gradients of the model parameters to prevent accumulation of gradients from previous iterations. This can be done using the `grad.zero_()`, `requires_grad_(False)`, `detach()`, or `torch.no_grad()` methods in PyTorch. grad.zero_() is used to set the gradients of all model parameters to zero. This is typically done at the beginning of each training iteration before performing the backward pass to compute new gradients. requires_grad_(False) can be used to temporarily disable gradient tracking for specific tensors, which can be useful for performing operations that should not contribute to gradient calculations. detach() creates a new tensor that shares the same data but does not require gradients, effectively detaching it from the computational graph. torch.no_grad() is a context manager that temporarily disables gradient tracking for all operations within its block, which can be useful for inference or when performing operations that do not require gradient computation.

In [56]:
x = torch.tensor(2.0, requires_grad=True)
x.requires_grad_(False)  # This will stop tracking gradients for x
z = x ** 2  # This will not track gradients for z

x = torch.tensor(2.0, requires_grad=True)
z = x.detach() ** 2  # Detach z from the computation graph
z.backward()  # This will not compute gradients for x since z is detached

with torch.no_grad():
    x = torch.tensor(2.0, requires_grad=True)
    z = x ** 2  # This will not track gradients for z
    z.backward()  # This will not compute gradients for x since z is not tracking gradients

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn